# Entregável 1 — Especificação e Baseline

> **Grupo:** Renato Bueno Domingos de Oliveira
> **Tema/Projeto:** Assistente para Análise de Documentos Semânticos Epidemiológicos
> **Data:** 07/09/2026

Este notebook deve conter a **especificação inicial do sistema**, a implementação de um
**baseline funcional**, casos de teste e uma análise crítica das limitações observadas.

Use células **Markdown** para documentação, justificativas e análise.
Use células **Python** para configuração, implementação, experimentos, testes e coleta de resultados.

**Antes de enviar:** execute o notebook do início ao fim (`Runtime → Run all`) e salve com as
saídas visíveis. Um notebook sem saídas não permite avaliar o baseline.

# 1. Descrição do problema

Dados de vigilância epidemiológica, como os disponibilizados pelo **[SINAN](https://datasus.saude.gov.br/transferencia-de-arquivos/# )** (Sistema de Informação de Agravos de Notificação), são organizados predominantemente em formato tabular e contêm variáveis codificadas, categóricas e distribuídas entre diferentes dimensões epidemiológicas.

A exploração desses dados exige conhecimento da estrutura das tabelas e, frequentemente, **múltiplas operações de filtragem, agrupamento e agregação**. Além disso, algumas perguntas epidemiológicas não podem ser respondidas pela recuperação de um único valor, pois exigem a combinação e a síntese de evidências provenientes de **diferentes dimensões, como localização, tempo, características clínicas, desfechos e sorotipos.**

Por exemplo, uma pergunta como *"Quantos casos de dengue foram registrados em São Paulo?"* pode ser respondida por uma **consulta determinística**. Por outro lado, uma pergunta como "*Qual é o panorama epidemiológico da dengue em São Paulo?"* exige recuperar e integrar diferentes evidências antes de produzir uma **resposta em linguagem natural.**

Este projeto investiga o uso de representações semânticas intermediárias, denominadas **Visões Semânticas Materializadas (VSM)**, para organizar evidências derivadas dos dados epidemiológicos e apoiar sistemas de perguntas e respostas baseados em modelos de linguagem.

Nesta primeira versão será implementado um **baseline simples**, baseado em uma única chamada a um modelo de linguagem, utilizando como contexto um documento semântico integrado previamente selecionado. O objetivo é estabelecer uma referência inicial para comparação com arquiteturas posteriores que poderão incorporar recuperação semântica, ferramentas, **ontologias** e agentes.




# 2. Usuário-alvo e stakeholders

**Usuário principal:** analista ou pesquisador que necessita consultar e interpretar informações epidemiológicas sem conhecer detalhadamente a estrutura dos bancos de dados ou as codificações utilizadas.

**Usuários secundários:** profissionais de vigilância epidemiológica, gestores de saúde, estudantes e pesquisadores que necessitam explorar indicadores de arboviroses.

**Stakeholders relevantes:**
- instituições de vigilância epidemiológica;
- gestores de saúde pública;
- pesquisadores da área de **saúde e computação**;
- responsáveis pela manutenção e integração das fontes de dados.

**Objetivos dos usuários:**
- formular perguntas em linguagem natural;
- recuperar informações epidemiológicas relevantes;
- obter sínteses envolvendo múltiplas dimensões;
- identificar as evidências utilizadas na resposta;
- reduzir a necessidade de conhecer diretamente os códigos e atributos das fontes tabulares.

# 3. Casos de uso principais

### UC-01 — Consulta determinística

- **Ator:** analista epidemiológico.
- **Entrada:** pergunta em linguagem natural, por exemplo:  
  *"Quantas notificações de dengue foram registradas no Brasil no período analisado de 2026?"*
- **Objetivo:** obter um indicador epidemiológico objetivo presente no documento.
- **Saída esperada:** resposta contendo o valor correspondente, o período de referência e a evidência utilizada.
- **Situação de sucesso:** o sistema retorna corretamente o valor presente no contexto fornecido, sem alterar números, períodos ou unidades.

### UC-02 — Consulta interpretativa

- **Ator:** analista ou pesquisador.
- **Entrada:** pergunta em linguagem natural, por exemplo:  
  *"O que o panorama indica sobre o comportamento temporal das notificações de dengue no período analisado de 2026?"*
- **Objetivo:** interpretar evidências temporais presentes no documento.
- **Saída esperada:** síntese textual baseada exclusivamente nos indicadores fornecidos, considerando, quando relevante, média semanal, semana de pico e número de notificações no pico.
- **Situação de sucesso:** a resposta interpreta corretamente os indicadores temporais disponíveis, sem inferir tendências ou comportamentos que não estejam sustentados pelo documento.

### UC-03 — Consulta multidimensional

- **Ator:** analista ou pesquisador.
- **Entrada:** pergunta em linguagem natural, por exemplo:  
  *"Qual é o panorama epidemiológico integrado da dengue no Brasil no período analisado de 2026?"*
- **Objetivo:** integrar evidências provenientes de diferentes dimensões epidemiológicas.
- **Saída esperada:** síntese envolvendo, quando relevantes, evidências temporais, geográficas, virológicas, clínicas e de desfechos.
- **Situação de sucesso:** o sistema combina corretamente as evidências disponíveis em uma resposta coerente, preservando números, percentuais e limitações descritas no documento, sem extrapolar o conteúdo fornecido.

### UC-04 — Informação ausente ou fora do escopo

- **Ator:** qualquer usuário.
- **Entrada:** pergunta cuja resposta não esteja presente no documento ou que não pertença ao escopo definido para o sistema. Por exemplo:  
  *"Como prevenir a dengue?"*
- **Objetivo:** verificar se o sistema reconhece a ausência de evidência suficiente e evita utilizar conhecimento externo.
- **Saída esperada:** indicação explícita de que os documentos fornecidos não contêm informação suficiente para responder ou de que a questão está fora do escopo do sistema.
- **Situação de sucesso:** o sistema se abstém de fornecer uma resposta factual não sustentada pelas evidências disponíveis.

# 4. Escopo, não-objetivos e premissas

| Item | Definição |
|---|---|
| **Escopo** | Perguntas em português sobre dengue, utilizando como contexto o documento semântico integrado **Panorama Epidemiológico Integrado da Dengue no Brasil — 2026**. Uma pergunta por execução. |
| **Não-objetivos** | Não serão implementados nesta versão agentes, ontologias, grafos de conhecimento, busca vetorial, execução automática de SQL, atualização incremental das VSM ou integração dinâmica com múltiplas fontes. |
| **Premissas** | O documento semântico utilizado como contexto foi previamente produzido e validado; seu conteúdo cabe na janela de contexto do modelo; os valores de referência dos testes são derivados do próprio documento. |

# 5. Entradas e saídas

## Entradas

O baseline recebe:

1. uma pergunta em linguagem natural, em português;
2. um documento semântico integrado utilizado como contexto.

O documento contém, de forma simplificada:

- identificador;
- fonte;
- doença;
- ano;
- escopo temporal;
- indicadores epidemiológicos agregados;
- informações temporais, geográficas, clínicas, virológicas e de desfechos;
- observações sobre limitações dos dados.

## Saídas

O sistema produz uma resposta em linguagem natural contendo:

- resposta à pergunta;
- síntese das evidências relevantes;
- indicação do documento semântico utilizado;
- indicação explícita de ausência de informação quando a resposta não puder ser sustentada pelo contexto fornecido.

# 6. Requisitos funcionais


| | |
|--|--|
|**RF-01:** o sistema deve receber uma pergunta epidemiológica em português e retornar uma resposta textual. |
 |**Verificação:** execução automática dos casos de teste.|
|**RF-02:** em perguntas determinísticas, a resposta deve conter corretamente os valores presentes no documento semântico fornecido
 |**Verificação:** comparação automática com respostas de referência.|
|**RF-03:** em perguntas interpretativas, a resposta deve utilizar somente informações presentes no contexto.|
 |**Verificação:** inspeção manual das afirmações e comparação com as VSM utilizadas.|
|**RF-04:** em perguntas multidimensionais, o sistema deve sintetizar evidências de pelo menos duas dimensões quando elas forem necessárias e estiverem disponíveis.|
 |**Verificação:** cobertura dos elementos esperados no conjunto de teste.|
|**RF-05:** quando a informação solicitada não estiver disponível nas VSM, o sistema deve declarar que não há evidência suficiente para responder.|
  |**Verificação:** casos de teste específicos de informação ausente.
|**RF-06:** a resposta deve indicar os identificador do documento semântico utilizado como evidência.|
  |**Verificação:** presença dos identificadores esperados na saída.



# 7. Requisitos não funcionais e restrições

| ID | Requisito | Descrição | Como verificar |
|---|---|---|---|
| **RNF-01** | **Correção** | O sistema deverá atingir inicialmente pelo menos **80% de respostas corretas** no conjunto de avaliação. | Comparar as respostas produzidas com as respostas de referência e calcular a proporção de acertos. |
| **RNF-02** | **Fidelidade às evidências** | As afirmações factuais relevantes deverão estar sustentadas por informações contidas nos documentos fornecidos. | Conferir se cada afirmação factual relevante possui evidência correspondente nos documentos utilizados. |
| **RNF-03** | **Abstenção** | Quando a informação necessária não estiver disponível, o sistema deverá reconhecer a insuficiência de evidência. | Executar casos de teste com informação ausente e verificar se o sistema evita produzir uma resposta não sustentada. |
| **RNF-04** | **Latência** | O tempo de execução de cada consulta deverá ser registrado. | Medir e registrar, em segundos, o tempo total de cada consulta. |
| **RNF-05** | **Reprodutibilidade** | As execuções deverão registrar modelo, versão, temperatura, versão do prompt e data da execução. | Verificar se todos os metadados de execução foram armazenados junto aos resultados. |
| **RNF-06** | **Custo computacional** | O número de chamadas ao modelo deverá ser registrado e, quando disponível, também os tokens de entrada e saída e o custo aproximado. | Registrar chamadas, tokens e custo por caso de teste e calcular os valores agregados ao final da execução. |

# 8. Recursos externos potencialmente necessários

Nesta versão do baseline são utilizados os seguintes recursos:

| Recurso | Situação atual | Finalidade |
|---|---|---|
| **Groq API** | Utilizado no baseline | Execução do modelo de linguagem |
| **Modelo `openai/gpt-oss-20b`** | Utilizado no baseline | Geração das respostas estruturadas |
| **LangChain / langchain-groq** | Utilizado no baseline | Integração com o modelo e saída estruturada |
| **Pydantic** | Utilizado no baseline | Definição e validação do formato de saída |
| **Pandas** | Utilizado no baseline | Organização dos resultados experimentais |
| **Documento semântico `SINAN_DENGUE_2026_PANORAMA_GERAL`** | Utilizado no baseline | Fonte de evidência epidemiológica |
| **Banco vetorial** | Hipótese para versões futuras | Recuperação automática de documentos relevantes |
| **Ontologia de aplicação** | Hipótese para versões futuras | Apoio à recuperação e interpretação semântica |
| **Agentes especializados** | Hipótese para versões futuras | Divisão da análise por dimensão epidemiológica |
| **Ferramentas ou consultas estruturadas** | Hipótese para versões futuras | Acesso a evidências não presentes no documento integrado |

Nesta primeira versão não são utilizados servidores MCP, memória persistente,
planejamento ou múltiplos agentes.



# 9. Tipo de baseline escolhido

O baseline é classificado como **parcial**.

Nesta primeira versão, o sistema executa a tarefa central de responder perguntas
epidemiológicas em linguagem natural utilizando como contexto um único documento
semântico integrado: o **Panorama Epidemiológico Integrado da Dengue no Brasil
— 2026**.

A escolha é adequada porque permite avaliar isoladamente a capacidade do modelo
de linguagem de localizar, interpretar e sintetizar informações já presentes em
um documento semântico, antes da introdução de mecanismos mais complexos de
recuperação e seleção de documentos.

Nesta versão foram excluídos:

- busca vetorial;
- seleção automática de documentos semânticos;
- consulta à ontologia;
- planejamento;
- múltiplos agentes;
- consulta direta às bases tabulares.

O baseline servirá como referência para verificar posteriormente se mecanismos
de recuperação semântica, ferramentas e agentes especializados melhoram a
correção, a cobertura e a rastreabilidade das respostas.

# 10. Critérios preliminares de sucesso

| Critério | Requisito associado | Como será medido |
|---|---|---|
| Correção factual | RF-02 / RNF-01 | Comparação com os valores esperados na resposta de referência |
| Cobertura | RF-04 | Proporção dos elementos esperados presentes na resposta |
| Fidelidade às evidências | RF-03 / RNF-02 | Verificação de que as afirmações são sustentadas pelo documento |
| Abstenção | RF-05 / RNF-03 | Identificação correta dos casos de informação ausente |
| Rastreabilidade | RF-06 | Presença do identificador do documento utilizado |
| Latência | RNF-04 | Mediana do tempo de execução das consultas |
| Reprodutibilidade | RNF-05 | Registro da configuração de cada execução |
| Tokens e custo | RNF-06 | Tokens de entrada/saída, chamadas e custo quando disponível |

# 11. Configuração do ambiente



In [20]:
%pip install -q -U langchain langchain-groq pydantic pandas

In [21]:
import os, getpass, datetime, platform

def carregar_chave_groq() -> str:
    """Funciona no Colab (userdata) e localmente (variável de ambiente)."""
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"
    try:
        from google.colab import userdata          # noqa: F401
        #os.environ["GROQ_API_KEY"] = userdata.get("INF0093-2026-2S")
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        return "Colab userdata"
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
        return "entrada manual"

origem = carregar_chave_groq()
assert os.environ.get("GROQ_API_KEY"), "Chave não configurada."
print("Chave carregada via:", origem)

Chave carregada via: variável de ambiente


In [22]:
from langchain_groq import ChatGroq

# MODEL_NAME = "llama-3.3-70b-versatile"   # Meta
MODEL_NAME = "openai/gpt-oss-20b"          # OpenAI

TEMPERATURE = 0
PROMPT_VERSAO = "v1"

llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

# Registro da execução: acompanha os resultados até o Entregável 4.
#
RUN_INFO = {
    "modelo": MODEL_NAME,
    "temperatura": TEMPERATURE,
    "prompt_versao": PROMPT_VERSAO,
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
}
RUN_INFO

{'modelo': 'openai/gpt-oss-20b',
 'temperatura': 0,
 'prompt_versao': 'v1',
 'data': '2026-09-07T13:07:59',
 'python': '3.13.15'}

# 12. Dados ou entradas de exemplo



In [23]:
# Exemplo
# sample_document = "texto do documento de entrada"
# sample_query = "pergunta representativa do problema"

call_document = """
# Panorama epidemiológico integrado da dengue no Brasil - 2026

## Identificação

- ID do documento: SINAN_DENGUE_2026_PANORAMA_GERAL
- Tipo de documento: panorama_epidemiologico_integrado
- Domínio: epidemiologico
- Fonte: SINAN
- Doença: dengue

## Escopo

- Localização: Brasil
- Ano: 2026
- Semanas epidemiológicas: 1 a 34
- Total de semanas: 34

## Situação geral

- Total de notificações: 444.266
- Período: semanas epidemiológicas 1 a 34

## Comportamento temporal

- Média semanal de notificações: 13.066,65
- Mediana semanal de notificações: 11.983,50
- Semana de pico: 15
- Notificações na semana de pico: 21.470
- Percentual do total concentrado na semana de pico: 4,83%

## Distribuição geográfica

- UF com maior número de notificações por residência: GO
- Total de notificações nessa UF: 108.912
- Município com maior número de notificações: Goiânia
- Total de notificações no município: 31.778
- Município com maior incidência: Turvânia
- Maior incidência observada: 9.084,82 por 100 mil habitantes

## Perfil virológico

- Registros com sorotipo informado: 29.517 (6,64%)
- Registros sem sorotipo informado: 414.749 (93,36%)
- UFs sem sorotipo informado: 1

## Perfil clínico

- Sinal clínico mais frequente: Febre (88,21%)
- Doença preexistente mais frequente: Hipertensão arterial (7,74%)

## Hospitalização e evolução

- Registros avaliáveis para hospitalização: 325.931
- Hospitalizados: 27.589 (8,46%)
- Registros com evolução avaliável: 309.618
- Evolução predominante: Cura (99,82%)

## Perfil dos registros classificados em categorias de óbito

- Total de registros classificados em categorias de óbito: 559
- Categoria predominante: Óbito pelo agravo (54,20%)
- Registros com sorotipo informado nesse subconjunto: 111
- Sorotipo predominante entre os registros com sorotipo informado:
  DENV-2 (69,37%)
"""

## 12.1 - Definindo a saída estruturada

In [24]:
from pydantic import BaseModel, Field
from typing import Literal, List


class EpidemiologicalAnalysis(BaseModel):

    answer: str = Field(
        description="Resposta direta à pergunta epidemiológica."
    )

    evidence: List[str] = Field(
        description=(
            "Trechos ou evidências extraídos exclusivamente "
            "do documento fornecido."
        )
    )

    document_id: str = Field(
        description="Identificador do documento utilizado."
    )

    confidence: Literal["alta", "media", "baixa"] = Field(
        description=(
            "Confiança da resposta considerando apenas "
            "as evidências disponíveis no documento."
        )
    )

    insufficient_information: bool = Field(
        description=(
            "True quando o documento não contém evidência "
            "suficiente para responder."
        )
    )

# 13. Implementação do baseline



## 13.1 - Prompt do baseline

In [25]:
SYSTEM_V1 = """
Você é um assistente especializado em análise de documentos
epidemiológicos.

Sua tarefa é responder à pergunta utilizando EXCLUSIVAMENTE
as informações presentes no documento fornecido.

Regras:

1. Não utilize conhecimento externo.
2. Não invente valores, relações ou interpretações.
3. Sempre forneça evidências presentes no documento.
4. Preserve números, percentuais, períodos e unidades.
5. Considere explicitamente as limitações descritas no documento.
6. Se a informação necessária não estiver disponível,
   indique isso explicitamente.
7. Nesse caso:
   - insufficient_information = true;
   - não tente completar a resposta com conhecimento externo.
8. document_id deve identificar o documento utilizado.

A resposta deve seguir o esquema estruturado definido.
"""

structured_llm = llm.with_structured_output(
    EpidemiologicalAnalysis,
    include_raw=True
)

## 13.2 - Função do baseline

In [26]:
import time
from langchain_core.messages import SystemMessage, HumanMessage


def baseline(pergunta: str):

    mensagem_usuario = f"""
DOCUMENTO:

{call_document}

PERGUNTA:

{pergunta}
"""

    inicio = time.perf_counter()

    retorno = structured_llm.invoke([
        SystemMessage(content=SYSTEM_V1),
        HumanMessage(content=mensagem_usuario)
    ])

    latencia = time.perf_counter() - inicio

    resultado = retorno["parsed"]
    raw = retorno["raw"]

    usage = getattr(raw, "usage_metadata", {}) or {}

    metricas = {
        "latencia_s": round(latencia, 3),
        "tokens_entrada": usage.get("input_tokens"),
        "tokens_saida": usage.get("output_tokens"),
        "tokens_total": usage.get("total_tokens"),
        "chamadas_llm": 1,
    }

    return resultado, metricas

# 14. Conjunto de avaliação


## 14.1 - Primeiro teste manual

Antes de criar os 10 testes, um teste simples

In [27]:
resultado, metricas = baseline(
    "Qual foi o total de notificações de dengue no Brasil em 2026?"
)

print("RESPOSTA")
print(resultado.answer)

print("\nEVIDÊNCIAS")
for evidencia in resultado.evidence:
    print("-", evidencia)

print("\nDOCUMENTO")
print(resultado.document_id)

print("\nCONFIANÇA")
print(resultado.confidence)

print("\nINFORMAÇÃO INSUFICIENTE")
print(resultado.insufficient_information)

print("\nMÉTRICAS")
print(metricas)

RESPOSTA
444.266

EVIDÊNCIAS
- Total de notificações: 444.266

DOCUMENTO
SINAN_DENGUE_2026_PANORAMA_GERAL

CONFIANÇA
alta

INFORMAÇÃO INSUFICIENTE
False

MÉTRICAS
{'latencia_s': 0.455, 'tokens_entrada': 905, 'tokens_saida': 136, 'tokens_total': 1041, 'chamadas_llm': 1}


A resposta de referência para esse caso deve ser 444.266 notificações, considerando as semanas epidemiológicas 1 a 34.

# 14.2 - O conjunto de testes congelado

O conteúdo real do panorama da dengue, e não perguntas fictícias. As perguntas consideram as semanas epidemiológicas 1 a 34 de acordo com o SINAN. A semana 34 corresponde a última data de notificação que é 23/08/2026 a 29/08/2026

In [42]:
test_cases = [

    {
        "id": "T01",
        "dimensao": "geral",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Quantas notificações de dengue foram consideradas "
            "no Brasil em 2026?"
        ),
        "esperado": ["444.266"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T02",
        "dimensao": "temporal",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Em qual semana epidemiológica ocorreu o pico "
            "de notificações e quantas notificações foram registradas?"
        ),
        "esperado": ["15", "21.470"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T03",
        "dimensao": "geografica",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual UF apresentou o maior número de notificações "
            "por residência?"
        ),
        "esperado": ["GO"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T04",
        "dimensao": "geografica",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual município apresentou a maior incidência "
            "de dengue?"
        ),
        "esperado": ["Turvânia"],
        "cobertura_minima": 1.0,
    },


    {
        "id": "T05",
        "dimensao": "virologica",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual percentual dos registros apresentou "
            "sorotipo informado?"
        ),
        "esperado": ["6,64%"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T06",
        "dimensao": "clinica",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual foi o sinal clínico mais frequente?"
        ),
        "esperado": ["Febre", "88,21%"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T07",
        "dimensao": "desfechos",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual foi a evolução predominante entre "
            "os registros avaliáveis?"
        ),
        "esperado": ["Cura"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T08",
        "dimensao": "multidimensional",
        "tipo": "composto",
        "verificacao": "auto",
        "pergunta": (
            "Resuma o panorama temporal, geográfico e clínico "
            "da dengue no Brasil em 2026."
        ),
        "esperado": [
            "semana 15",
            "GO",
            "Febre"
        ],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T09",
        "dimensao": "qualidade_dados",
        "tipo": "interpretativo",
        "verificacao": "manual",
        "pergunta": (
            "Que limitação deve ser considerada ao interpretar "
            "os dados de sorotipo?"
        ),
        "esperado": [
            "93,36%",
            "ausência"
        ],
        "cobertura_minima": 0.5,
    },

    {
        "id": "T10",
        "dimensao": "fora_escopo",
        "tipo": "informacao_ausente",
        "verificacao": "auto",
        "pergunta": (
            "Qual medicamento deve ser utilizado para tratar "
            "a dengue?"
        ),
        "esperado": None,
        "cobertura_minima": None,
    },
]

print(len(test_cases), "casos definidos.")

10 casos definidos.


Os valores esperados desses casos estão diretamente documentados no panorama. Por exemplo, o documento registra 29.517 casos com sorotipo informado, equivalentes a 6,64%, e 414.749 sem informação de sorotipo, equivalentes a 93,36%.
### Avaliação manual do caso interpretativo T09

O caso T09 possui resposta aberta e será avaliado manualmente segundo três
critérios:

| Critério | Pontuação |
|---|---:|
| Identifica que há baixa disponibilidade de informação de sorotipo | 1 |
| Sustenta a interpretação com informação presente no documento | 1 |
| Não introduz informação externa ou conclusão não sustentada | 1 |

O caso será considerado correto se atingir pelo menos 2 dos 3 critérios.

Depois de ler a resposta completa do T09, será registrado:

```
avaliacao_manual = {
    "T09": {
        "baixa_disponibilidade_sorotipo": True,
        "fundamentada_documento": True,
        "sem_extrapolacao": True,
        "pontuacao": 3,
        "aprovado_manual": True
    }
}

avaliacao_manual
```

Não será marcado isso como True automaticamente agora: primeiro eu irei ler a  resposta completa e irei avaliar os três itens.

# 15. Implementação da verificação

A verificação do baseline combina critérios determinísticos e inspeção dos campos
estruturados retornados pelo modelo.

Para os casos com resposta objetiva, a avaliação verifica se os elementos
esperados aparecem na resposta produzida. Para os casos de informação ausente,
é verificado se o sistema marcou corretamente a necessidade de abstenção.

Também são registrados indicadores de cobertura dos elementos esperados e
presença de evidências documentais.

In [43]:
import re
import unicodedata


def normalizar_texto(texto):
    """
    Normaliza texto para facilitar comparações automáticas.
    Remove acentos, converte para minúsculas e reduz espaços.
    """
    if texto is None:
        return ""

    texto = str(texto).lower()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()


def verificar_elementos_esperados(
    resposta,
    esperados
):
    """
    Verifica quantos elementos esperados aparecem na resposta.
    """

    if esperados is None:
        return None, None

    resposta_norm = normalizar_texto(resposta)

    encontrados = []

    for esperado in esperados:

        esperado_norm = normalizar_texto(
            esperado
        )

        encontrado = (
            esperado_norm
            in resposta_norm
        )

        encontrados.append(
            encontrado
        )

    cobertura = (
        sum(encontrados)
        /
        len(encontrados)
    )

    return encontrados, cobertura


def verificar_caso(caso, resultado):
    """
    Avalia automaticamente os casos objetivos e sinaliza
    os casos que exigem avaliação manual.
    """

    # --------------------------------------------------------
    # AVALIAÇÃO MANUAL
    # --------------------------------------------------------

    if caso.get("verificacao") == "manual":

        return {
            "aprovado": None,
            "cobertura": None,
            "elementos_encontrados": None,
            "abstencao_correta": None,
            "possui_evidencias": bool(resultado.evidence),
            "avaliacao": "manual"
        }

    # --------------------------------------------------------
    # INFORMAÇÃO AUSENTE
    # --------------------------------------------------------

    if caso["tipo"] == "informacao_ausente":

        aprovado = (
            resultado.insufficient_information
            is True
        )

        return {
            "aprovado": aprovado,
            "cobertura": None,
            "elementos_encontrados": None,
            "abstencao_correta": aprovado,
            "possui_evidencias": bool(resultado.evidence),
            "avaliacao": "automatica"
        }

    # --------------------------------------------------------
    # CASOS OBJETIVOS
    # --------------------------------------------------------

    encontrados, cobertura = (
        verificar_elementos_esperados(
            resultado.answer,
            caso["esperado"]
        )
    )

    cobertura_minima = caso.get(
        "cobertura_minima",
        1.0
    )

    aprovado = (
        cobertura is not None
        and cobertura >= cobertura_minima
        and resultado.insufficient_information is False
    )

    return {
        "aprovado": aprovado,
        "cobertura": cobertura,
        "elementos_encontrados": encontrados,
        "abstencao_correta": None,
        "possui_evidencias": bool(resultado.evidence),
        "avaliacao": "automatica"
    }

Uma verificação de rastreabilidade, porque foi definido que a saída precisa indicar o documento utilizado:

In [44]:
DOCUMENTO_ESPERADO = (
    "SINAN_DENGUE_2026_PANORAMA_GERAL"
)


def verificar_rastreabilidade(
    resultado
):
    """
    Verifica se o modelo identificou corretamente
    o documento utilizado.
    """

    recebido = normalizar_texto(
        resultado.document_id
    )

    esperado = normalizar_texto(
        DOCUMENTO_ESPERADO
    )

    return recebido == esperado

# 16. Experimentos

O baseline será executado sobre o conjunto congelado de casos de teste.

Para cada consulta serão registrados:

- resposta produzida;
- evidências retornadas;
- documento utilizado;
- indicação de informação insuficiente;
- cobertura dos elementos esperados;
- aprovação automática;
- latência;
- número de chamadas ao modelo;
- tokens de entrada e saída, quando disponíveis.

Os mesmos casos deverão ser reutilizados nas futuras versões da arquitetura
para permitir comparação direta com o baseline.

In [45]:
registros = []

for caso in test_cases:

    print(
        f'Executando {caso["id"]}: '
        f'{caso["pergunta"]}'
    )

    # --------------------------------------------------------
    # EXECUTA BASELINE
    # --------------------------------------------------------

    resultado, metricas = baseline(
        caso["pergunta"]
    )

    # --------------------------------------------------------
    # VERIFICAÇÃO
    # --------------------------------------------------------

    verificacao = verificar_caso(
        caso,
        resultado
    )

    rastreabilidade_ok = (
        verificar_rastreabilidade(
            resultado
        )
    )

    # --------------------------------------------------------
    # REGISTRO
    # --------------------------------------------------------

    registro = {

        "id": caso["id"],

        "dimensao": caso.get(
            "dimensao"
        ),

        "tipo": caso["tipo"],

        "pergunta": caso["pergunta"],

        "resposta": resultado.answer,

        "esperado": caso.get(
            "esperado"
        ),

        "evidencias": resultado.evidence,

        "documento": resultado.document_id,

        "confianca": resultado.confidence,

        "informacao_insuficiente":
            resultado.insufficient_information,

        "avaliacao":
            verificacao["avaliacao"],

        "cobertura":
            verificacao["cobertura"],

        "aprovado":
            verificacao["aprovado"],

        "possui_evidencias":
            verificacao["possui_evidencias"],

        "abstencao_correta":
            verificacao["abstencao_correta"],

        "rastreabilidade_ok":
            rastreabilidade_ok,

        "latencia_s":
            metricas["latencia_s"],

        "tokens_entrada":
            metricas.get(
                "tokens_entrada"
            ),

        "tokens_saida":
            metricas.get(
                "tokens_saida"
            ),

        "tokens_total":
            metricas.get(
                "tokens_total"
            ),

        "chamadas_llm":
            metricas["chamadas_llm"],
    }

    registros.append(
        registro
    )

    print(
        "Aprovado:",
        registro["aprovado"]
    )

    print(
        "Latência:",
        registro["latencia_s"],
        "s"
    )

    print("-" * 60)

Executando T01: Quantas notificações de dengue foram consideradas no Brasil em 2026?
Aprovado: True
Latência: 0.433 s
------------------------------------------------------------
Executando T02: Em qual semana epidemiológica ocorreu o pico de notificações e quantas notificações foram registradas?
Aprovado: True
Latência: 0.474 s
------------------------------------------------------------
Executando T03: Qual UF apresentou o maior número de notificações por residência?
Aprovado: True
Latência: 0.405 s
------------------------------------------------------------
Executando T04: Qual município apresentou a maior incidência de dengue?
Aprovado: True
Latência: 0.578 s
------------------------------------------------------------
Executando T05: Qual percentual dos registros apresentou sorotipo informado?
Aprovado: True
Latência: 0.453 s
------------------------------------------------------------
Executando T06: Qual foi o sinal clínico mais frequente?
Aprovado: True
Latência: 0.462 s
-----

# 17. Resultados

Os 10 casos de teste foram executados utilizando o mesmo documento semântico,
modelo e configuração do baseline. Os resultados a seguir apresentam a resposta
produzida, o tipo de avaliação, a cobertura dos elementos esperados, a presença
de evidências, a rastreabilidade, a latência e o consumo de tokens.

Os casos objetivos e de informação ausente foram avaliados automaticamente.
O caso interpretativo T09 foi separado para avaliação manual

In [53]:
import pandas as pd


df_resultados = pd.DataFrame(
    registros
)
# ============================================================
# CONFIGURAÇÃO DE VISUALIZAÇÃO
# ============================================================

pd.set_option(
    "display.max_colwidth",
    None
)

pd.set_option(
    "display.max_columns",
    None
)

display(
    df_resultados
)



,id,dimensao,tipo,pergunta,resposta,esperado,evidencias,documento,confianca,informacao_insuficiente,avaliacao,cobertura,aprovado,possui_evidencias,abstencao_correta,rastreabilidade_ok,latencia_s,tokens_entrada,tokens_saida,tokens_total,chamadas_llm
0,T01,geral,normal,Quantas notificações de dengue foram consideradas no Brasil em 2026?,444.266 notificações de dengue foram consideradas no Brasil em 2026.,[444.266],[- Total de notificações: 444.266],SINAN_DENGUE_2026_PANORAMA_GERAL,alta,False,automatica,1.0,True,True,None,True,0.433,904,142,1046,1
1,T02,temporal,normal,Em qual semana epidemiológica ocorreu o pico de notificações e quantas notificações foram registradas?,"A semana epidemiológica de pico foi a semana 15, com 21.470 notificações registradas.","[15, 21.470]","[Semana de pico: 15, Notificações na semana de pico: 21.470]",SINAN_DENGUE_2026_PANORAMA_GERAL,alta,False,automatica,1.0,True,True,None,True,0.474,909,233,1142,1
2,T03,geografica,normal,Qual UF apresentou o maior número de notificações por residência?,GO,[GO],[UF com maior número de notificações por residência: GO],SINAN_DENGUE_2026_PANORAMA_GERAL,alta,False,automatica,1.0,True,True,None,True,0.405,901,125,1026,1
3,T04,geografica,normal,Qual município apresentou a maior incidência de dengue?,Turvânia,[Turvânia],"[Município com maior incidência: Turvânia, Maior incidência observada: 9.084,82 por 100 mil habitantes]",SINAN_DENGUE_2026_PANORAMA_GERAL,alta,False,automatica,1.0,True,True,None,True,0.578,899,168,1067,1
4,T05,virologica,normal,Qual percentual dos registros apresentou sorotipo informado?,"6,64%","[6,64%]","[Registros com sorotipo informado: 29.517 (6,64%)]",SINAN_DENGUE_2026_PANORAMA_GERAL,alta,False,automatica,1.0,True,True,None,True,0.453,898,159,1057,1
5,T06,clinica,normal,Qual foi o sinal clínico mais frequente?,"Febre (88,21%)","[Febre, 88,21%]","[Sinal clínico mais frequente: Febre (88,21%)]",SINAN_DENGUE_2026_PANORAMA_GERAL,alta,False,automatica,1.0,True,True,None,True,0.462,899,143,1042,1
6,T07,desfechos,normal,Qual foi a evolução predominante entre os registros avaliáveis?,Cura,[Cura],"[Registros com evolução avaliável: 309.618, Evolução predominante: Cura (99,82%)]",SINAN_DENGUE_2026_PANORAMA_GERAL,alta,False,automatica,1.0,True,True,None,True,0.310,901,146,1047,1
7,T08,multidimensional,composto,"Resuma o panorama temporal, geográfico e clínico da dengue no Brasil em 2026.","Em 2026, a dengue no Brasil apresentou um panorama epidemiológico marcado por um total de 444 266 notificações entre as semanas epidemiológicas 1 a 34, com média semanal de 13 066,65 e mediana de 11 983,50. O pico de incidência ocorreu na semana 15, com 21 470 notificações, representando 4,83 % do total anual. Geograficamente, o estado de Goiás (GO) registrou o maior número de casos por residência (108 912), sendo Goiânia o município com mais notificações (31 778). O município com maior incidência foi Turvânia, com 9 084,82 casos por 100 000 habitantes. Clinicamente, a febre foi o sintoma mais frequente (88,21 %) e a hipertensão arterial a doença preexistente mais comum (7,74 %).","[semana 15, GO, Febre]","[Média semanal de notificações: 13.066,65, Mediana semanal de notificações: 11.983,50, Semana de pico: 15, Notificações na semana de pico: 21.470, Percentual do total concentrado na semana de pico: 4,83%, UF com maior número de notificações por residência: GO, Total de notificações nessa UF: 108.912, Município com maior número de notificações: Goiânia, Total de notificações no município: 31.778, Município com maior incidência: Turvânia, Maior incidência observada: 9.084,82 por 100 mil habitantes, Sinal clínico mais frequente: Febre (88,21%), Doença preexistente mais frequente: Hipertensão arterial (7,74%)]",SINAN_DENGUE_2026_PANORAMA_GERAL,alta,False,automatica,1.0,True,True,None,True,0.714,909,447,1356,1
8,T09,qualidade_dados,interpretativo,Que limitação deve ser considerada ao interpretar os dados de sorotipo?,"Ao interpretar os dados de sorotipo, deve‑se considerar qu

Visualização somente das respostas

In [55]:
pd.set_option(
    "display.max_colwidth",
    None
)
display(
    df_resultados[
        [
            "id",
            "pergunta",
            "resposta",
            "evidencias"
        ]
    ]
)

,id,pergunta,resposta,evidencias
0,T01,Quantas notificações de dengue foram consideradas no Brasil em 2026?,444.266 notificações de dengue foram consideradas no Brasil em 2026.,[- Total de notificações: 444.266]
1,T02,Em qual semana epidemiológica ocorreu o pico de notificações e quantas notificações foram registradas?,"A semana epidemiológica de pico foi a semana 15, com 21.470 notificações registradas.","[Semana de pico: 15, Notificações na semana de pico: 21.470]"
2,T03,Qual UF apresentou o maior número de notificações por residência?,GO,[UF com maior número de notificações por residência: GO]
3,T04,Qual município apresentou a maior incidência de dengue?,Turvânia,"[Município com maior incidência: Turvânia, Maior incidência observada: 9.084,82 por 100 mil habitantes]"
4,T05,Qual percentual dos registros apresentou sorotipo informado?,"6,64%","[Registros com sorotipo informado: 29.517 (6,64%)]"
5,T06,Qual foi o sinal clínico mais frequente?,"Febre (88,21%)","[Sinal clínico mais frequente: Febre (88,21%)]"
6,T07,Qual foi a evolução predominante entre os registros avaliáveis?,Cura,"[Registros com evolução avaliável: 309.618, Evolução predominante: Cura (99,82%)]"
7,T08,"Resuma o panorama temporal, geográfico e clínico da dengue no Brasil em 2026.","Em 2026, a dengue no Brasil apresentou um panorama epidemiológico marcado por um total de 444 266 notificações entre as semanas epidemiológicas 1 a 34, com média semanal de 13 066,65 e mediana de 11 983,50. O pico de incidência ocorreu na semana 15, com 21 470 notificações, representando 4,83 % do total anual. Geograficamente, o estado de Goiás (GO) registrou o maior número de casos por residência (108 912), sendo Goiânia o município com mais notificações (31 778). O município com maior incidência foi Turvânia, com 9 084,82 casos por 100 000 habitantes. Clinicamente, a febre foi o sintoma mais frequente (88,21 %) e a hipertensão arterial a doença preexistente mais comum (7,74 %).","[Média semanal de notificações: 13.066,65, Mediana semanal de notificações: 11.983,50, Semana de pico: 15, Notificações na semana de pico: 21.470, Percentual do total concentrado na semana de pico: 4,83%, UF com maior número de notificações por residência: GO, Total de notificações nessa UF: 108.912, Município com maior número de notificações: Goiânia, Total de notificações no município: 31.778, Município com maior incidência: Turvânia, Maior incidência observada: 9.084,82 por 100 mil habitantes, Sinal clínico mais frequente: Febre (88,21%), Doença preexistente mais frequente: Hipertensão arterial (7,74%)]"
8,T09,Que limitação deve ser considerada ao interpretar os dados de sorotipo?,"Ao interpretar os dados de sorotipo, deve‑se considerar que apenas 6,64 % dos registros possuem sorotipo informado, enquanto 93,36 % não têm essa informação. Essa alta taxa de ausência de sorotipo pode tornar os resultados não representativos da realidade epidemiológica, pois a maioria dos casos não contribui para a análise de distribuição de sorotipos.","[Registros com sorotipo informado: 29.517 (6,64%), Registros sem sorotipo informado: 414.749 (93,36%)]"
9,T10,Qual medicamento deve ser utilizado para tratar a dengue?,Não há informação no documento sobre o medicamento a ser utilizado para tratar a dengue.,[]


In [56]:
for _, row in df_resultados.iterrows():

    print("=" * 80)
    print(f'CASO: {row["id"]}')
    print("=" * 80)

    print("\nPERGUNTA:")
    print(row["pergunta"])

    print("\nRESPOSTA:")
    print(row["resposta"])

    print("\nEVIDÊNCIAS:")
    for evidencia in row["evidencias"]:
        print(f"- {evidencia}")

    print()

CASO: T01

PERGUNTA:
Quantas notificações de dengue foram consideradas no Brasil em 2026?

RESPOSTA:
444.266 notificações de dengue foram consideradas no Brasil em 2026.

EVIDÊNCIAS:
- - Total de notificações: 444.266

CASO: T02

PERGUNTA:
Em qual semana epidemiológica ocorreu o pico de notificações e quantas notificações foram registradas?

RESPOSTA:
A semana epidemiológica de pico foi a semana 15, com 21.470 notificações registradas.

EVIDÊNCIAS:
- Semana de pico: 15
- Notificações na semana de pico: 21.470

CASO: T03

PERGUNTA:
Qual UF apresentou o maior número de notificações por residência?

RESPOSTA:
GO

EVIDÊNCIAS:
- UF com maior número de notificações por residência: GO

CASO: T04

PERGUNTA:
Qual município apresentou a maior incidência de dengue?

RESPOSTA:
Turvânia

EVIDÊNCIAS:
- Município com maior incidência: Turvânia
- Maior incidência observada: 9.084,82 por 100 mil habitantes

CASO: T05

PERGUNTA:
Qual percentual dos registros apresentou sorotipo informado?

RESPOSTA:
6,6

Como a tabela inteira pode ficar larga demais foi criada uma visão resumida:

In [47]:
COLUNAS_RESUMO = [
    "id",
    "dimensao",
    "tipo",
    "avaliacao",
    "aprovado",
    "cobertura",
    "possui_evidencias",
    "rastreabilidade_ok",
    "informacao_insuficiente",
    "latencia_s",
    "tokens_total"
]

display(
    df_resultados[
        COLUNAS_RESUMO
    ]
)

,id,dimensao,tipo,avaliacao,aprovado,cobertura,possui_evidencias,rastreabilidade_ok,informacao_insuficiente,latencia_s,tokens_total
0,T01,geral,normal,automatica,True,1.0,True,True,False,0.433,1046
1,T02,temporal,normal,automatica,True,1.0,True,True,False,0.474,1142
2,T03,geografica,normal,automatica,True,1.0,True,True,False,0.405,1026
3,T04,geografica,normal,automatica,True,1.0,True,True,False,0.578,1067
4,T05,virologica,normal,automatica,True,1.0,True,True,False,0.453,1057
5,T06,clinica,normal,automatica,True,1.0,True,True,False,0.462,1042
6,T07,desfechos,normal,automatica,True,1.0,True,True,False,0.310,1047
7,T08,multidimensional,composto,automatica,True,1.0,True,True,False,0.714,1356
8,T09,qualidade_dados,interpretativo,manual,None,NaN,True,True,False,7.984,1262
9,T10,fora_escopo,informacao_ausente,automatica,True,NaN,False,True,True,8.613,999


Agora calculamos os indicadores gerais.

In [48]:
# ============================================================
# RESULTADOS GERAIS
# ============================================================

total_casos = len(
    df_resultados
)

# ------------------------------------------------------------
# CASOS COM AVALIAÇÃO AUTOMÁTICA
# ------------------------------------------------------------

df_automaticos = df_resultados[
    df_resultados["avaliacao"] == "automatica"
].copy()

total_automaticos = len(
    df_automaticos
)

aprovados_automaticos = int(
    df_automaticos[
        "aprovado"
    ].sum()
)

taxa_aprovacao_automatica = (
    aprovados_automaticos
    /
    total_automaticos
    *
    100
)

# ------------------------------------------------------------
# CASOS COM AVALIAÇÃO MANUAL
# ------------------------------------------------------------

total_manuais = int(
    (
        df_resultados["avaliacao"]
        == "manual"
    ).sum()
)

# ------------------------------------------------------------
# LATÊNCIA
# ------------------------------------------------------------

latencia_mediana = (
    df_resultados[
        "latencia_s"
    ].median()
)

latencia_media = (
    df_resultados[
        "latencia_s"
    ].mean()
)

# ------------------------------------------------------------
# EVIDÊNCIAS
#
# Consideramos apenas casos respondíveis.
# O caso de informação ausente não precisa apresentar evidência.
# ------------------------------------------------------------

df_respondiveis = df_resultados[
    df_resultados[
        "informacao_insuficiente"
    ] == False
]

taxa_evidencias = (
    df_respondiveis[
        "possui_evidencias"
    ].mean()
    *
    100
)

# ------------------------------------------------------------
# RASTREABILIDADE
# ------------------------------------------------------------

taxa_rastreabilidade = (
    df_resultados[
        "rastreabilidade_ok"
    ].mean()
    *
    100
)

# ------------------------------------------------------------
# CHAMADAS AO LLM
# ------------------------------------------------------------

total_chamadas = (
    df_resultados[
        "chamadas_llm"
    ].sum()
)

E o resumo:

In [49]:
RESUMO = {

    "total_casos":
        total_casos,

    "casos_avaliacao_automatica":
        total_automaticos,

    "casos_automaticos_aprovados":
        aprovados_automaticos,

    "taxa_aprovacao_automatica_pct":
        round(
            taxa_aprovacao_automatica,
            2
        ),

    "casos_avaliacao_manual":
        total_manuais,

    "latencia_media_s":
        round(
            latencia_media,
            3
        ),

    "latencia_mediana_s":
        round(
            latencia_mediana,
            3
        ),

    "respostas_com_evidencias_pct":
        round(
            taxa_evidencias,
            2
        ),

    "rastreabilidade_correta_pct":
        round(
            taxa_rastreabilidade,
            2
        ),

    "total_chamadas_llm":
        int(
            total_chamadas
        )
}

RESUMO

{'total_casos': 10,
 'casos_avaliacao_automatica': 9,
 'casos_automaticos_aprovados': 9,
 'taxa_aprovacao_automatica_pct': 100.0,
 'casos_avaliacao_manual': 1,
 'latencia_media_s': np.float64(2.043),
 'latencia_mediana_s': 0.468,
 'respostas_com_evidencias_pct': np.float64(100.0),
 'rastreabilidade_correta_pct': np.float64(100.0),
 'total_chamadas_llm': 10}

Tokens disponíveis:

In [50]:
if (
    "tokens_total"
    in df_resultados.columns
):

    tokens_validos = (
        df_resultados[
            "tokens_total"
        ]
        .dropna()
    )

    if len(tokens_validos) > 0:

        RESUMO[
            "tokens_total"
        ] = int(
            tokens_validos.sum()
        )

RESUMO

{'total_casos': 10,
 'casos_avaliacao_automatica': 9,
 'casos_automaticos_aprovados': 9,
 'taxa_aprovacao_automatica_pct': 100.0,
 'casos_avaliacao_manual': 1,
 'latencia_media_s': np.float64(2.043),
 'latencia_mediana_s': 0.468,
 'respostas_com_evidencias_pct': np.float64(100.0),
 'rastreabilidade_correta_pct': np.float64(100.0),
 'total_chamadas_llm': 10,
 'tokens_total': 11044}

Tabela final mais bonita

Uma tabela específica para apresentação:

In [51]:
# ============================================================
# TABELA FINAL DE APRESENTAÇÃO
# ============================================================

df_apresentacao = (
    df_resultados[
        [
            "id",
            "dimensao",
            "tipo",
            "avaliacao",
            "aprovado",
            "cobertura",
            "latencia_s"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# FORMATA COBERTURA
# ------------------------------------------------------------

df_apresentacao[
    "cobertura"
] = (
    df_apresentacao[
        "cobertura"
    ]
    .apply(
        lambda x:
        f"{x * 100:.0f}%"
        if pd.notna(x)
        else "-"
    )
)


# ------------------------------------------------------------
# DEFINE RÓTULO DO RESULTADO
# ------------------------------------------------------------

def rotulo_resultado(row):

    if row["avaliacao"] == "manual":
        return "Avaliação manual"

    if row["aprovado"] is True:
        return "Aprovado"

    if row["aprovado"] is False:
        return "Reprovado"

    return "-"


df_apresentacao[
    "resultado"
] = (
    df_apresentacao.apply(
        rotulo_resultado,
        axis=1
    )
)


# ------------------------------------------------------------
# SELECIONA COLUNAS FINAIS
# ------------------------------------------------------------

df_apresentacao = (
    df_apresentacao[
        [
            "id",
            "dimensao",
            "tipo",
            "resultado",
            "cobertura",
            "latencia_s"
        ]
    ]
)


display(
    df_apresentacao
)

,id,dimensao,tipo,resultado,cobertura,latencia_s
0,T01,geral,normal,Aprovado,100%,0.433
1,T02,temporal,normal,Aprovado,100%,0.474
2,T03,geografica,normal,Aprovado,100%,0.405
3,T04,geografica,normal,Aprovado,100%,0.578
4,T05,virologica,normal,Aprovado,100%,0.453
5,T06,clinica,normal,Aprovado,100%,0.462
6,T07,desfechos,normal,Aprovado,100%,0.310
7,T08,multidimensional,composto,Aprovado,100%,0.714
8,T09,qualidade_dados,interpretativo,Avaliação manual,-,7.984
9,T10,fora_escopo,informacao_ausente,Aprovado,-,8.613


## Avaliação manual do caso T09

O caso T09 possui natureza interpretativa e, por isso, não é avaliado apenas
por correspondência literal entre termos esperados e resposta produzida.

A avaliação manual considera os seguintes critérios:

| Critério | Pontuação máxima |
|---|---:|
| Identifica corretamente a baixa disponibilidade de informação de sorotipo | 1 |
| Fundamenta a interpretação em informações presentes no documento | 1 |
| Não introduz informação externa ou conclusão não sustentada | 1 |

A pontuação máxima é 3 pontos.

O caso é considerado aprovado quando obtém pelo menos 2 pontos.

In [52]:
avaliacao_manual = {
    "T09": {
        "avaliador": "Renato Bueno Domingos de Oliveira",

        "criterios": {
            "identifica_baixa_disponibilidade_sorotipo": 1,
            "fundamentada_no_documento": 1,
            "sem_extrapolacao": 1
        },

        "pontuacao": 3,
        "pontuacao_maxima": 3,
        "aprovado_manual": True,

        "observacao": (
            "A resposta reconhece a baixa disponibilidade de informação "
            "de sorotipo, fundamenta a interpretação nas evidências do "
            "documento e não introduz informação externa."
        )
    }
}

registro_manual = avaliacao_manual["T09"]

df_avaliacao_manual = pd.DataFrame(
    [
        {
            "id": "T09",

            "criterio_1":
                registro_manual["criterios"][
                    "identifica_baixa_disponibilidade_sorotipo"
                ],

            "criterio_2":
                registro_manual["criterios"][
                    "fundamentada_no_documento"
                ],

            "criterio_3":
                registro_manual["criterios"][
                    "sem_extrapolacao"
                ],

            "pontuacao":
                registro_manual["pontuacao"],

            "maximo":
                registro_manual["pontuacao_maxima"],

            "resultado":
                (
                    "Aprovado"
                    if registro_manual["aprovado_manual"]
                    else "Reprovado"
                )
        }
    ]
)

display(df_avaliacao_manual)

,id,criterio_1,criterio_2,criterio_3,pontuacao,maximo,resultado
0,T09,1,1,1,3,3,Aprovado


# 18. Análise crítica do baseline

O baseline implementado atende ao objetivo principal desta primeira etapa:
receber uma pergunta epidemiológica em linguagem natural, utilizar como contexto
um documento semântico previamente selecionado e produzir uma resposta estruturada
com evidências e identificação do documento utilizado.

Nos casos objetivos, o sistema apresentou bom desempenho para perguntas gerais,
temporais, geográficas, virológicas, clínicas e de desfechos. O caso
multidimensional também foi respondido adequadamente, demonstrando que uma única
chamada ao modelo é capaz de combinar informações de diferentes dimensões quando
essas evidências estão presentes no mesmo documento semântico.

O requisito de rastreabilidade foi atendido em todas as execuções, uma vez que
o sistema identificou corretamente o documento
`SINAN_DENGUE_2026_PANORAMA_GERAL` utilizado como contexto. As respostas
factuais para as quais havia informação disponível também foram acompanhadas
de evidências extraídas do documento.

O caso de informação ausente mostrou que o baseline é capaz de reconhecer quando
o documento não contém evidência suficiente e se abster de responder com
conhecimento externo. Esse comportamento é importante para reduzir respostas
não fundamentadas no contexto epidemiológico fornecido.

Entretanto, algumas limitações foram observadas. A principal limitação
arquitetural é que o documento relevante já é fornecido diretamente ao modelo.
Assim, o baseline não precisa decidir quais documentos consultar nem recuperar
automaticamente evidências em uma coleção maior de documentos semânticos. Dessa
forma, o experimento atual avalia principalmente a capacidade de interpretação
do documento fornecido, e não a capacidade de recuperação de informação.

Também foi identificada uma limitação no processo de avaliação automática.
Casos interpretativos não são adequadamente avaliados apenas por correspondência
literal de termos esperados. No caso T09, uma resposta semanticamente correta
poderia utilizar palavras diferentes das definidas no conjunto de referência.
Por esse motivo, esse caso foi submetido a uma avaliação manual baseada em uma
rubrica com três critérios.

Outra limitação diz respeito ao tamanho do conjunto experimental. Foram utilizados
dez casos de teste e cada caso foi executado uma única vez. Portanto, diferenças
observadas em latência entre consultas simples, multidimensionais e interpretativas
não permitem concluir que determinado tipo de pergunta seja necessariamente mais
custoso.

As limitações observadas podem ser separadas em três grupos. A limitação de
recuperação automática decorre da arquitetura do baseline. Eventuais variações
na forma de redação das respostas decorrem do comportamento do modelo de
linguagem. Já dificuldades relacionadas à correspondência literal entre resposta
e ground truth decorrem do método de avaliação utilizado, e não necessariamente
de um erro do sistema.

Assim, o baseline funciona adequadamente como referência inicial, mas ainda não
resolve o problema completo de consulta sobre uma coleção distribuída de
documentos semânticos epidemiológicos.

# 19. Possíveis evoluções arquiteturais

A evolução arquitetural deve ser orientada pelas limitações observadas no baseline,
evitando a introdução de componentes que não produzam benefício mensurável.

## Recuperação automática de documentos

A principal evolução necessária é a introdução de um mecanismo de recuperação
automática de documentos semânticos. No baseline atual, o documento relevante
é fornecido diretamente ao modelo. Em uma versão posterior, a pergunta do usuário
deverá ser utilizada para localizar automaticamente os documentos mais relevantes
em uma coleção maior.

Essa evolução pode ser implementada inicialmente por meio de recuperação semântica,
por exemplo com embeddings e busca vetorial. Posteriormente, metadados e conceitos
da ontologia de aplicação poderão ser utilizados para tornar a recuperação mais
específica.

O principal custo dessa evolução é o aumento da complexidade da arquitetura,
incluindo indexação, recuperação, seleção de contexto e novas métricas de avaliação.

## Workflow

Um workflow explícito pode ser útil caso o processamento passe a envolver etapas
distintas, como identificação do tipo de pergunta, seleção da dimensão
epidemiológica, recuperação de evidências e geração da resposta final.

Essa abordagem pode melhorar a rastreabilidade do processo, mas aumenta o número
de etapas e potencialmente o número de chamadas ao modelo.

## Ferramentas e ReAct

O uso de ferramentas pode ser considerado caso algumas perguntas exijam operações
que não possam ser resolvidas apenas com os documentos recuperados, como cálculos,
consultas estruturadas ou acesso a dados específicos.

Uma abordagem ReAct pode permitir que o modelo decida quando utilizar essas
ferramentas. Entretanto, essa estratégia introduz maior custo computacional,
mais chamadas ao modelo e maior dificuldade de controle do fluxo de execução.

## Memória

Memória persistente não é considerada necessária para a próxima versão, pois
o cenário experimental atual processa uma pergunta por execução e não depende
de histórico conversacional. Sua introdução só seria justificada se o sistema
passasse a suportar consultas encadeadas que dependessem do contexto de perguntas
anteriores.

## MCP

O uso de MCP também não é necessário nesta etapa. Ele poderia ser considerado
no futuro caso fosse necessário padronizar o acesso a fontes externas,
serviços ou ferramentas distribuídas. No cenário atual, isso acrescentaria
complexidade sem resolver diretamente a principal limitação observada.

## Planejamento

Planejamento autônomo pode ser útil em consultas complexas compostas por várias
subtarefas. Entretanto, os resultados atuais ainda não demonstram necessidade
desse recurso. Antes de introduzi-lo, deve-se avaliar se um workflow explícito
e mais simples já é suficiente.

## Múltiplos agentes

Uma arquitetura multiagente poderá ser avaliada em uma etapa posterior caso as
consultas multidimensionais demonstrem necessidade de especialização.

Uma possibilidade seria utilizar agentes especializados nas dimensões temporal,
geográfica, clínica, virológica e de desfechos, coordenados por um agente
responsável pela integração das evidências.

Essa arquitetura teria como principal benefício potencial a especialização e
a decomposição de perguntas complexas. Entretanto, seu custo seria maior em
termos de chamadas ao modelo, latência, coordenação e avaliação.

Por esse motivo, a adoção de múltiplos agentes só será considerada vantajosa se
apresentar ganhos mensuráveis em relação ao baseline e a arquiteturas intermediárias
mais simples.

# 20. Pergunta obrigatória

> **Como o grupo pretende demonstrar, ao final do curso, que a arquitetura final
> apresenta vantagens em relação ao baseline?**

A comparação entre o baseline e as versões posteriores será realizada utilizando
o mesmo conjunto congelado de casos de teste, sempre que aplicável, mantendo
também o mesmo modelo de linguagem e o mesmo formato de saída. Dessa forma,
as diferenças observadas poderão ser atribuídas principalmente às alterações
arquiteturais.

A hipótese inicial é que uma arquitetura capaz de recuperar e combinar
automaticamente documentos semânticos relevantes será capaz de responder a um
conjunto mais amplo de perguntas epidemiológicas do que o baseline, especialmente
quando a evidência estiver distribuída em documentos diferentes.

A principal vantagem esperada não é apenas melhorar respostas que o baseline já
consegue produzir, mas permitir a resolução de casos que o baseline não consegue
processar porque depende da seleção manual de um único documento de contexto.

A comparação será realizada por meio das seguintes métricas:

- correção das respostas;
- cobertura dos elementos esperados;
- fidelidade às evidências;
- capacidade de abstenção quando a informação estiver ausente;
- rastreabilidade das evidências utilizadas;
- taxa de sucesso nos casos de teste;
- latência por consulta;
- número de chamadas ao modelo;
- quantidade de tokens de entrada e saída;
- custo aproximado, quando disponível.

Além dos casos utilizados neste baseline, versões posteriores poderão incluir
casos em que a resposta dependa de documentos distintos ou de evidências
específicas por dimensão epidemiológica.

A arquitetura final será considerada vantajosa somente se apresentar ganhos
mensuráveis de cobertura, correção ou capacidade de resolver consultas mais
complexas, sem introduzir custos de latência, chamadas ao modelo ou consumo de
tokens desproporcionais aos benefícios obtidos.

# 21. Conclusão

Este entregável definiu e implementou um baseline para um assistente de análise
de documentos semânticos epidemiológicos aplicado ao panorama da dengue no
Brasil em 2026.

O problema abordado consiste em permitir que usuários formulem perguntas
epidemiológicas em linguagem natural e recebam respostas fundamentadas em
evidências presentes em documentos semânticos.

O baseline foi implementado como uma solução simples baseada em uma única chamada
a um modelo de linguagem, utilizando como contexto o documento semântico
`SINAN_DENGUE_2026_PANORAMA_GERAL`.

A solução foi classificada como baseline parcial, pois executa a tarefa central
de interpretação e resposta, mas depende da seleção prévia do documento relevante
e não realiza recuperação automática em uma coleção de documentos.

Os experimentos contemplaram dez casos de teste envolvendo consultas gerais,
temporais, geográficas, virológicas, clínicas, de desfechos, multidimensionais,
interpretativas e de informação ausente.

Os resultados mostraram que o baseline é adequado para consultas cuja evidência
está explicitamente presente no documento fornecido. Também foi possível verificar
rastreabilidade das respostas, uso de evidências, capacidade de abstenção e
métricas de execução como latência, chamadas ao modelo e consumo de tokens.

O experimento também demonstrou que perguntas interpretativas exigem mecanismos
de avaliação diferentes da simples correspondência lexical, motivo pelo qual
foi utilizada uma avaliação manual com rubrica específica para o caso T09.

A principal limitação identificada é a necessidade de fornecer previamente o
documento correto ao modelo. Dessa forma, a próxima evolução do sistema deverá
investigar mecanismos de recuperação automática de documentos semânticos,
mantendo o baseline atual como referência experimental.

A hipótese para a próxima versão é que a recuperação automática de evidências
permitirá ampliar a cobertura das consultas e responder a perguntas cuja informação
esteja distribuída entre diferentes documentos, preservando correção,
rastreabilidade e fundamentação das respostas.

---

# Checklist antes da entrega

- [x] O problema está claramente definido.
- [x] O usuário-alvo foi identificado.
- [x] Escopo e não-objetivos estão explícitos.
- [x] Existem requisitos funcionais **verificáveis**.
- [x] Existem requisitos não funcionais.
- [x] Cada critério de sucesso diz **como** será medido.
- [x] O tipo de baseline foi classificado e justificado.
- [x] O baseline executa sem erros.
- [x] Existem pelo menos três casos de teste, cobrindo mais de um tipo.
- [x] Modelo, temperatura e data da execução estão registrados.
- [x] Latência, tokens e número de chamadas foram registrados.
- [x] Os resultados estão na tabela e interpretados.
- [x] As limitações foram analisadas.
- [x] A pergunta obrigatória foi respondida.
- [x] O notebook foi executado do início ao fim e **salvo com as saídas**.
- [x] O notebook pode ser executado por outra pessoa.
- [x] Nenhuma chave de API foi incluída no notebook.